# AutoGluon Time Series — `target_1ay_hiz`

Bu notebook, bir sonraki ayın otomobil devir hızını tahmin eden nihai IP-6 çalışmasını yeniden üretir.

- AutoGluon ayarı: `medium_quality`
- Tahmin ufku: 1 ay
- Puanlanan çıktı: `h=1` medyan tahmini
- Doğrulama originleri: 2020-02 – 2025-05, toplam 64 ay
- Test originleri: 2025-06 – 2026-05; **bu notebookta açılmaz**


In [1]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Notebook notebooks/ klasöründen veya proje kökünden açılabilir.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").exists(), "Proje kökü bulunamadı."

try:
    import autogluon.timeseries as agts
except ImportError as exc:
    raise RuntimeError(
        "Bu notebook'u projenin .venv-ag Python ortamıyla çalıştırın."
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Proje:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("AutoGluon:", agts.__version__)


Proje: C:\Users\YOGA\Desktop\araç piyasasında yön analizi için kullanılan yöntemler v2
Python: 3.12.7
AutoGluon: 1.6.1


## Targetın anlamı

Gerçekleşme ayı `τ` için:

\[
target\_1ay\_hiz_{\tau}
=100\ln\left(\frac{V_{\tau}}{V_{\tau-1}}\right)
\]

Origin `t` tarihinde AutoGluon `target_1ay_hiz[t+1]` değerini tahmin eder.

- Pozitif: gelecek ay hacim artışı.
- Negatif: gelecek ay hacim düşüşü.
- Mutlak değer: logaritmik hareketin şiddeti.


In [2]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "birlesik_target_setleri"
    / "target_1ay_hiz_tum_featurelar_final.csv"
)

data = pd.read_csv(DATA_PATH)
data["referans_ayi"] = pd.to_datetime(data["referans_ayi"], errors="raise")
data = data.sort_values("referans_ayi").reset_index(drop=True)

expected_months = pd.date_range("2018-02-01", "2026-06-01", freq="MS")

assert data.shape == (101, 14)
assert data["referans_ayi"].equals(pd.Series(expected_months, name="referans_ayi"))
assert not data["referans_ayi"].duplicated().any()
assert data["target_1ay_hiz"].notna().all()
assert np.isfinite(data["target_1ay_hiz"]).all()

print("Tarih:", data["referans_ayi"].min().date(), "→", data["referans_ayi"].max().date())
print("Gözlem:", len(data))
display(data.head(3))


Tarih: 2018-02-01 → 2026-06-01
Gözlem: 101


,referans_ayi,arabam_reel_aylik_degisim_pct,tufe_aylik_degisim,betam_dom_gun,indicata_satis_ilan_orani_pct,odmd_hta_adet,osd_kamyonet_adet,indicata_perakende_fiyat_aylik_pct,indicata_satisa_donen_adet,osd_binek_kamyonet_toplam_adet,betam_satis_orani_pct,noter_devir_otomobil_adet,betam_talep_aylik_pct,target_1ay_hiz
0,2018-02-01,NaN,0.7317,NaN,NaN,"11,108.0000","39,738.0000",NaN,NaN,"133,851.0000",NaN,"419,533.0000",NaN,-5.9505
1,2018-03-01,NaN,0.9935,NaN,NaN,"16,547.0000","45,190.0000",NaN,NaN,"150,877.0000",NaN,"478,648.0000",NaN,13.1823
2,2018-04-01,NaN,1.8723,NaN,NaN,"16,018.0000","38,848.0000",NaN,NaN,"129,637.0000",NaN,"479,866.0000",NaN,0.2541


## Target formülünü doğrulama

İlk target 2018-01 hacmini gerektirdiği için dosyanın ikinci satırından itibaren yeniden hesaplanır.


In [3]:
volume = data["noter_devir_otomobil_adet"]
reconstructed = 100 * np.log(volume / volume.shift(1))
check_mask = reconstructed.notna()

max_target_difference = (
    reconstructed[check_mask] - data.loc[check_mask, "target_1ay_hiz"]
).abs().max()

assert max_target_difference < 1e-9
print("Formül doğrulandı. En büyük fark:", max_target_difference)


Formül doğrulandı. En büyük fark: 7.105427357601002e-15


## Deney kolları

- **T0:** yalnız target geçmişi.
- **T1:** beş geniş kapsamlı feature; tamamı `lag1`.

T1 feature’ları:

1. Noter otomobil devri
2. TÜFE aylık değişimi
3. OSD kamyonet adedi
4. OSD binek + kamyonet toplamı
5. ODMD hafif ticari araç adedi

`lag1`, hedef ayı `t+1` için yalnız karar ayında (`t`) bilinen değeri verir. Kaydırılmamış `lag0` sütun yasaktır.


In [4]:
T1_FEATURES = [
    "noter_devir_otomobil_adet",
    "tufe_aylik_degisim",
    "osd_kamyonet_adet",
    "osd_binek_kamyonet_toplam_adet",
    "odmd_hta_adet",
]

source = data.set_index("referans_ayi")[T1_FEATURES].copy()
source["odmd_hta_adet"] = source["odmd_hta_adet"].ffill()

lagged = source.shift(1)
lagged.columns = [f"{column}_lag1" for column in lagged.columns]

assert all(column.endswith("_lag1") for column in lagged.columns)
assert not set(T1_FEATURES).intersection(lagged.columns)

sample_origin = pd.Timestamp("2024-04-01")
target_month = sample_origin + pd.DateOffset(months=1)

for original in T1_FEATURES:
    supplied = lagged.loc[target_month, f"{original}_lag1"]
    observed_at_origin = source.loc[sample_origin, original]
    assert np.isclose(supplied, observed_at_origin, rtol=0, atol=1e-12)

print("Örnek lag1 denetimi geçti:", sample_origin.date(), "→", target_month.date())


Örnek lag1 denetimi geçti: 2024-04-01 → 2024-05-01


## Eğitimi çalıştırma veya tamamlanmış checkpointi kullanma

`RUN_TRAINING=False` mevcut sonuçları yükler. Baştan veya yarım kalan eğitim için `True` yapın; `RESUME=True` tamamlanmış originleri atlar.


In [5]:
def load_module(module_name: str, script_path: Path):
    """Bir Python betiğini fonksiyonlarına erişebileceğimiz modül olarak yükler."""
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


RUN_TRAINING = False
RESUME = True

runner = load_module(
    "ag_target_1ay_runner",
    PROJECT_ROOT / "scripts" / "ag_05_1ay_h1.py",
)

if RUN_TRAINING:
    model_data = runner.load_data()
    predictions = runner.run(model_data, resume=RESUME)
    evaluated = runner.scored(predictions)
    ranking = runner.ranking(evaluated)
    _, chosen_model = runner.select_model(ranking)
    paired, mcnemar = runner.paired_tables(evaluated, chosen_model)
    selection = runner.selection_text(ranking, chosen_model)
    verdict = runner.decision_text(ranking, paired, mcnemar, chosen_model)

    ranking.to_csv(runner.RANK_PATH, index=False, encoding="utf-8-sig")
    paired.to_csv(runner.PAIRED_PATH, index=False, encoding="utf-8-sig")
    mcnemar.to_csv(runner.MCNEMAR_PATH, index=False, encoding="utf-8-sig")
    runner.SELECTION_PATH.write_text(selection, encoding="utf-8")
    runner.DECISION_PATH.write_text(verdict, encoding="utf-8")
else:
    predictions = pd.read_csv(runner.PRED_PATH, parse_dates=["origin", "hedef_ay"])
    ranking = pd.read_csv(runner.RANK_PATH)
    paired = pd.read_csv(runner.PAIRED_PATH)
    mcnemar = pd.read_csv(runner.MCNEMAR_PATH)
    selection = runner.SELECTION_PATH.read_text(encoding="utf-8")
    verdict = runner.DECISION_PATH.read_text(encoding="utf-8")

print("Tahmin satırı:", len(predictions))


Tahmin satırı: 1536


## Güvenlik kontrolleri


In [6]:
assert predictions["origin"].min() == pd.Timestamp("2020-02-01")
assert predictions["origin"].max() == pd.Timestamp("2025-05-01")
assert predictions["origin"].nunique() == 64
assert len(predictions[predictions["origin"].ge("2025-06-01")]) == 0
assert predictions["hedef_ay"].eq(predictions["origin"] + pd.DateOffset(months=1)).all()
assert predictions["y_pred_q50"].notna().all()
assert np.isfinite(predictions["y_pred_q50"]).all()

leakage_audit = runner.LEAKAGE_PATH.read_text(encoding="utf-8")
assert "A7_A8_PASS=True" in leakage_audit

print("Test origin satırı: 0")
print("A7/A8 lag1 sızıntı denetimi geçti.")


Test origin satırı: 0
A7/A8 lag1 sızıntı denetimi geçti.


## Model seçimi ve sonuçlar

Model yalnız 2020-02–2022-12 arasındaki SELECT bloğunda seçilir. 2023 sonrası CONFIRM sonucuna bakılarak değiştirilmez.


In [7]:
display(
    ranking[
        ranking["rejim"].eq("TUM")
        & ranking["model"].isin(["Chronos2", "DirectTabular", "sifir", "naive"])
    ].sort_values("MAE")
)

display(paired)
display(mcnemar)
print(selection)
print(verdict)


,rejim,kol,model,n,MAE,RMSE,DA_adet,DA_yuzde,gercek_pozitif,gercek_negatif,cogunluk_taban_yuzde,DA_binom_p_vs_0_5
48,TUM,T1,Chronos2,64,18.0204,26.1373,38,59.3750,33,31,51.5625,0.0843
49,TUM,T0,sifir,64,18.7683,25.8236,0,0.0000,33,31,51.5625,1.0000
50,TUM,T1,sifir,64,18.7683,25.8236,0,0.0000,33,31,51.5625,1.0000
55,TUM,T0,Chronos2,64,19.6346,27.2971,27,42.1875,33,31,51.5625,0.9157
60,TUM,T1,DirectTabular,64,20.4258,26.3329,34,53.1250,33,31,51.5625,0.3540
61,TUM,T0,DirectTabular,64,20.4679,28.6499,36,56.2500,33,31,51.5625,0.1909
70,TUM,T0,naive,64,27.3742,37.8355,30,46.8750,33,31,51.5625,0.7338
71,TUM,T1,naive,64,27.3742,37.8355,30,46.8750,33,31,51.5625,0.7338


,rejim,karsilastirma,n,paired_win,paired_tie,candidate_MAE,reference_MAE,reference_eksi_candidate_MAE,net_hata_kazanci,en_buyuk_2_kazanc_cikarilinca_net_fark
0,SOK,T1_best_vs_sifir,35,19,0,23.5081,24.7130,1.2049,42.1711,18.5653
1,SOK,T1_best_vs_naive,35,20,0,23.5081,33.5223,10.0142,350.4971,205.0405
2,SOK,T1_best_vs_ayni_model_T0,35,21,0,23.5081,26.2949,2.7868,97.5386,69.7216
3,NORMAL,T1_best_vs_sifir,29,15,0,11.3974,11.5937,0.1963,5.6934,-21.1706
4,NORMAL,T1_best_vs_naive,29,21,0,11.3974,19.9541,8.5567,248.1437,168.2268
5,NORMAL,T1_best_vs_ayni_model_T0,29,14,0,11.3974,11.5964,0.1990,5.7706,-22.2139
6,TUM,T1_best_vs_sifir,64,34,0,18.0204,18.7683,0.7479,47.8645,19.6373
7,TUM,T1_best_vs_naive,64,41,0,18.0204,27.3742,9.3538,598.6408,453.1842
8,TUM,T1_best_vs_ayni_model_T0,64,35,0,18.0204,19.6346,1.6142,103.3092,72.9595


,rejim,candidate,reference,n,b_candidate_dogru_naive_yanlis,c_candidate_yanlis_naive_dogru,mcnemar_exact_p
0,SOK,T1_Chronos2,naive,35,11,11,1.0000
1,NORMAL,T1_Chronos2,naive,29,16,8,0.1516
2,TUM,T1_Chronos2,naive,64,27,19,0.3020


SECIM_BLOKU=SOK_2020-02_2022-12
SECILEN_KOL=T1
SECILEN_MODEL=Chronos2
SOK_MAE=23.508077
SOK_DA=19/35
NORMAL_MAE=11.397418
NORMAL_DA=19/29
NORMAL_SONUCA_BAKILARAK_MODEL_DEGISTIRILMEDI=True

SECILEN=T1_Chronos2
TUM_MAE=18.020434
TUM_DA=38/64 (59.3750%)
TUM_DA_BINOM_P=0.08432146
TUM_SIFIR_PAIRED_WIN=34/64
NORMAL_SIFIR_PAIRED_WIN=15/29
NAIVE_MCNEMAR_P=0.30199561
KRITERLER={'sifir_paired_win_en_az_40': False, 'sifira_gore_mae_en_az_yuzde10_iyi': False, 'en_buyuk_2_cikarilinca_net_pozitif': True, 'DA_en_az_40': False, 'naive_McNemar_p_altinda_005': False, 'normal_MAE_farki_pozitif': True, 'normal_paired_win_en_az_15': True}
IP6_KABUL=False



## Naive yön tanısı

Naive yön doğruluğu yaklaşık %50 ise targetta mekanik ters-yön problemi yoktur. Üç aylık targettaki çok düşük naive doğruluğunun aksine, bir aylık target bu kontrolde daha sağlıklıdır.


In [8]:
naive_direction = (
    ranking[ranking["model"].eq("naive")]
    [["rejim", "DA_adet", "n", "DA_yuzde"]]
    .drop_duplicates("rejim")
    .sort_values("rejim")
)
display(naive_direction)


,rejim,DA_adet,n,DA_yuzde
22,NORMAL,11,29,37.9310
46,SOK,19,35,54.2857
70,TUM,30,64,46.8750


## Deney kararı

Seçilen T1–Chronos2 modeli sıfır değişim tahmininden daha iyi görünse de ön kayıtlı hata, yön ve istatistiksel güven eşiklerinin tamamını geçemedi. Sonuç `IP6_KABUL=False`; test dönemi açılmadı.
